In [2]:
from google.colab import userdata
GIT_TOKEN=userdata.get('GITHUB_TOKEN')

In [ ]:
# Install the Vela compiler toolchain
!pip install ethos-u-vela

# Secure clone and pip install your custom codebase using GitHub Token credentials
import os
GIT_USERNAME = "bencejdanko"
GIT_REPO = f"github.com/{GIT_USERNAME}/fomo-people-counting.git"

!git clone https://{GIT_USERNAME}:{GIT_TOKEN}@{GIT_REPO}
%cd /content/fomo-people-counting
!pip install -e .

In [6]:
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from fomo_core.annotation import SamAnnotator
from fomo_core.pipeline import FomoDatasetBuilder
from fomo_core.model import build_and_compile_fomo
from fomo_core.quantization import convert_to_int8_tflite

# 1. Pipeline Setup
annotator = SamAnnotator()
builder = FomoDatasetBuilder(annotator=annotator)
raw_dataset = load_dataset("detection-datasets/coco", split="train", streaming=True)

# 2. Linear Dataset Ingestion
print("Extracting features...")
X_train_list, y_train_list = [], []
for example in raw_dataset.take(1000):  # Start scaling up datasets safely
    img_arr, grid_arr = builder.process_example(example)
    X_train_list.append(img_arr)
    y_train_list.append(grid_arr)

X_train, y_train = np.array(X_train_list), np.array(y_train_list)

# 3. TF Data Handlers
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(200).map(builder.synchronized_augment).batch(32).prefetch(tf.data.AUTOTUNE)

# 4. Engine Run
model = build_and_compile_fomo()
model.fit(train_ds, epochs=15)

# 5. INT8 Calibration
convert_to_int8_tflite(model, X_train)

# 6. Call the Vela compiler toolchain via bash
!vela fomo_production_int8.tflite \
  --accelerator-config ethos-u55-256 \
  --config configs/default_vela.ini \
  --memory-mode Shared_Sram \
  --system-config Ethos_U55_High_End_Embedded \
  --output-dir vela_output \
  --optimise Size

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/375M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

The image processor of type `SamImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


README.md:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

Extracting features...


TypeError: string indices must be integers, not 'str'